# Historical coursework archive

This is the original 2023–2024 coursework source. Saved outputs and personal/environment metadata have been removed. **The original scores are affected by data leakage.** The source retains those issues and the differences between preparation code and supplied data. Use the revised workflow in `src/` to run the current benchmark.

Use `../src/evaluate.py`, `../notebooks/project_walkthrough.ipynb`, and `../docs/REVIEW.md` for the corrected 2026 workflow. The original code and narrative below have not been silently rewritten.


In [ ]:
%pip install imbalanced-learn
%pip install folium pandas matplotlib
# Import the required packages
import pandas as pd
import numpy as np  
import matplotlib.pyplot as plt  
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from scipy.stats.mstats import winsorize
import warnings
import plotly.express as px
import plotly.offline as pyo
import plotly.graph_objs as go
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import LabelEncoder
import matplotlib.ticker as ticker
import folium
from folium.plugins import MarkerCluster
from folium import plugins
import plotly.express as px
from scipy.stats import mstats


In [ ]:
# Set the style for the plots
sns.set(style="whitegrid")

## 1. HR Analytics

### 1.1 Load and Explore the data

#### Statistical Analysis


###### In the initial phase of my analysis, I started by loading the HR Analytics dataset and conducted a statistical analysis to understand its structure and contents

In [ ]:
# Load the dataset
hr_data = pd.read_csv('hr_data.csv')

# Create a copy of the DataFrame for further analysis
hr_new = hr_data.copy()

###### The command hr_new.shape provides the overall structure of the HR dataset, detailing the number of rows and columns. It indicates the scope of the dataset.

In [ ]:
hr_new.shape

###### Using hr_new.head() and hr_new.tail(), I inspect the dataset's first and last entries to ensure consistency and correct formatting.

In [ ]:
# Show overview of the dataset
hr_new.head()

In [ ]:
hr_new.tail()

###### The hr_new.info() function provides a summary of the data set, highlighting data types and non-null counts. This step is  for identifying which features are categorical and require encoding and assessing the completeness of the data.

In [ ]:
# Basic information about the dataset
hr_new.info()

###### hr_new.describe() offers key statistics for numerical features, such as mean, standard deviation, and range. This informs me about the data's distribution, potential outliers, and whether feature scaling is necessary to normalize the data.

In [ ]:
# Descriptive statistics
hr_new.describe()

#### Numerical Data Analysis

###### In analyzing the HR Analytics dataset, I have narrowed the focus to key numerical features that are crucial for understanding what drives employee promotions. 

In [ ]:
hr_num = hr_new.select_dtypes(['int64', 'float64']).drop(columns='employee_id').copy()
hr_num.head()

###### In the numerical data analysis, I've classified features as binary (indicating simple yes/no conditions like KPIs met or awards won, crucial for promotion prospects) and non-binary (for more nuanced statistical scrutiny). This classification enables targeted analysis of each variable type's impact on promotion likelihood, with binary variables assessed for direct correlation and non-binary variables explored for deeper trends affecting career advancement.

In [ ]:
# Binary variables
binary_cols = [col for col in hr_num if hr_new[col].nunique() == 2]

# Non-binary numeric variables
non_binary_cols = [col for col in hr_num if col not in binary_cols]

###### In my analysis, I've used histograms and boxplots to analyze the distribution of non-binary numerical features. Histograms reveal distribution shapes, showing, for example, that 'number of trainings' is right-skewed, indicating fewer trainings per employee, and 'length of service' is left-skewed, with most employees having shorter tenures. Boxplots provide a summary of data distribution, median, and outliers, with 'age' being evenly distributed and outliers present in 'previous_year_rating' and 'length_of_service', pointing to unique cases in the workforce.

In [ ]:
# Plotting the distribution of non-binary numerical features
fig, axes = plt.subplots(len(non_binary_cols), 2, figsize=(12, 4 * len(non_binary_cols)))
for idx, col in enumerate(non_binary_cols):
    sns.histplot(hr_new[col], bins=30, kde=True, ax=axes[idx, 0])
    axes[idx, 0].set_title(f'Distribution of {col}')
    axes[idx, 0].set_ylabel('Count')
    
    sns.boxplot(x=hr_new[col], ax=axes[idx, 1])
    axes[idx, 1].set_title(f'Boxplot of {col}')
    axes[idx, 1].set_ylabel('Value')

plt.tight_layout()

###### I've charted the binary variables to gauge the balance of their values. The count plots show the distribution, crucial for spotting imbalances. Notably, 'is_promoted' displays a marked class imbalance, with promotions being much less common. This insight is vital, as such imbalances could skew the model's accuracy

In [ ]:
# Calculate the number of rows and columns for subplots
num_plots = len(binary_cols)
num_rows = (num_plots + 1) // 2  # Ensure at least 2 plots per row
num_cols = 2

# Create subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(12, 6*num_rows))
fig.suptitle('Distribution of Binary Numeric Features')

# Iterate through binary_cols and create count plots
for i, col in enumerate(binary_cols):
    row = i // num_cols
    col_num = i % num_cols
    ax = axes[row, col_num]
    sns.countplot(x=col, data=hr_new, ax=ax)
    ax.set_title(f'Distribution of {col}')
    ax.set_ylabel('Count')

# Remove any empty subplots
for i in range(num_plots, num_rows*num_cols):
    fig.delaxes(axes.flatten()[i])

# Adjust layout and display
plt.tight_layout()
plt.subplots_adjust(top=0.9)
plt.show()

#### Categorical Data Analysis

###### In analyzing categorical variables within the HR Analytics dataset, I've separated them for specific review, as their discrete nature demands an analytical method distinct from numerical data.

In [ ]:
hr_cat = hr_new.select_dtypes(['object']).copy()
hr_cat.head()

In [ ]:
hr_cat.describe()

###### I've utilized the nunique() function on the hr_cat DataFrame to determine the number of unique values within each categorical variabe which may be useful when doing encoding

In [ ]:
hr_cat.nunique()

###### Count plots of categorical variables reveal: "Sales & Marketing" has the highest employee count, hinting at departmental impacts on promotions; "region_2" houses a large employee base, suggesting regional effects; most employees have a bachelor's degree, a potential influencer on promotions; there's a gender skew, critical for bias prevention; and distinct recruitment channels are preferred, likely affecting promotion likelihood.

In [ ]:
# Number of categorical columns
n_cols = len(hr_cat.columns)

# Set the number of rows for subplots based on the number of columns
n_rows = int(np.ceil(n_cols / 2))

# Create a figure with subplots
plt.figure(figsize=(12 * 2, 6 * n_rows))  # Adjust the size as needed

for i, col in enumerate(hr_cat.columns):
    plt.subplot(n_rows, 2, i + 1)  # Adjust for number of rows and columns
    sns.countplot(x=col, data=hr_cat, order=sorted(hr_cat[col].dropna().unique()), palette='viridis')
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)

# Adjust layout for better fit
plt.tight_layout()
plt.show()

#### Univariate Analysis

###### Pie charts for KPIs, ratings, and awards underscore key trends: a small fraction surpass the 80% KPI mark, indicating notable performance; varied ratings with a tilt towards lower scores impact promotion evaluation; and the rarity of awards points to their significance in promotion considerations.

In [ ]:
# lets plot pie chart for the columns where we have very few categories
plt.rcParams['figure.figsize'] = (16,5)
plt.style.use('fivethirtyeight')
# plotting a pie chart to represent share of Previous year Rating of the Employees
plt.subplot(1, 3, 1)
labels = ['0','1']
sizes = hr_new['KPIs_met >80%'].value_counts()
colors = plt.cm.Wistia(np.linspace(0, 1, 5))
explode = [0, 0]
plt.pie(sizes, labels = labels, colors = colors, explode = explode, shadow = True, startangle = 90)
plt.title('KPIs Met > 80%', fontsize = 20)
# plotting a pie chart to represent share of Previous year Rating of the Employees
plt.subplot(1, 3, 2)
labels = ['1', '2', '3', '4', '5']
sizes = hr_new['previous_year_rating'].value_counts()
colors = plt.cm.Wistia(np.linspace(0, 1, 5))
explode = [0, 0, 0, 0, 0.1]
plt.pie(sizes, labels = labels, colors = colors, explode = explode, shadow = True, startangle = 90)
plt.title('Previous year Ratings', fontsize = 20)
# plotting a pie chart to represent share of Previous year Rating of the Employees
plt.subplot(1, 3, 3)
labels = ['0', '1']
sizes = hr_new['awards_won?'].value_counts()
colors = plt.cm.Wistia(np.linspace(0, 1, 5))
explode = [0,0.1]

plt.pie(sizes, labels = labels, colors = colors, explode = explode, shadow = True, startangle = 90)
plt.title('Awards Won?', fontsize = 20)
plt.legend()
plt.show()

###### Pie charts reveal the company's demographics: a majority have Bachelor's degrees, males outnumber females, and most hires come through standard channels over direct referrals. These findings are critical for modelling promotions and will inform the encoding of these variables.

In [ ]:
# lets plot pie chart for the columns where we have very few categories
plt.rcParams['figure.figsize'] = (16,5)

# plotting a pie chart to represent share of Previous year Rating of the Employees
plt.subplot(1, 3, 1)
labels = hr_new['education'].value_counts().index
sizes = hr_new['education'].value_counts()
colors = plt.cm.copper(np.linspace(0, 1, 5))
explode = [0, 0, 0.1]
plt.pie(sizes, labels = labels, colors = colors, explode = explode, shadow = True, startangle = 90)
plt.title('Education', fontsize = 20)
# plotting a pie chart to represent share of Previous year Rating of the Employees
plt.subplot(1, 3, 2)
labels = hr_new['gender'].value_counts().index
sizes = hr_new['gender'].value_counts()
colors = plt.cm.copper(np.linspace(0, 1, 5))
explode = [0, 0]
plt.pie(sizes, labels = labels, colors = colors, explode = explode, shadow = True, startangle = 90)
plt.title('Gender', fontsize = 20)
# plotting a pie chart to represent share of Previous year Rating of the Employees
plt.subplot(1, 3, 3)
labels = hr_new['recruitment_channel'].value_counts().index
sizes = hr_new['recruitment_channel'].value_counts()
colors = plt.cm.copper(np.linspace(0, 1, 5))
explode = [0,0,0.1]
plt.pie(sizes, labels = labels, colors = colors, explode=explode,shadow = True, startangle = 90)
plt.title('Recruitment Channel', fontsize = 20)
plt.show()

#### Bivariate Analysis

###### The stacked bar chart's bivariate analysis shows that despite fewer women in the workforce, their promotion rate equals that of men. This suggests gender doesn't significantly impact promotions, indicating a fair workplace. This is key for understanding career progression and maintaining a gender-unbiased predictive model.

In [ ]:
# Ignore warnings (if needed)
warnings.filterwarnings('ignore')

# Set figure size
plt.rcParams['figure.figsize'] = (15, 3)

# Create a cross-tabulation
x = pd.crosstab(hr_new['gender'], hr_new['is_promoted'])

# Define colors for the bars
colors = plt.cm.Wistia(np.linspace(0, 1, 5))

# Normalize the data and plot the stacked bar chart
x.div(x.sum(1).astype(float), axis=0).plot(kind='bar', stacked=False, color=colors)

# Set title and labels
plt.title('Effect of Gender on Promotion', fontsize=15)
plt.xlabel(' ')

# Show the plot
plt.show()

###### The area chart shows equal promotion rates across departments, suggesting departmental affiliation isn't a key predictor of promotions. This indicates either even distribution of promotions across departments or other factors being more influential in promotion decisions. This finding may reduce the importance of the department variable in predictive modeling for promotions.

In [ ]:
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (15,4)
x = pd.crosstab(hr_new['department'], hr_new['is_promoted'])
colors = plt.cm.copper(np.linspace(0, 1, 3))
x.div(x.sum(1).astype(float), axis = 0).plot(kind = 'area', stacked = False)
plt.title('Effect of Department on Promotion', fontsize = 15)
plt.xticks(rotation = 20)
plt.xlabel(' ')
plt.show()

###### The bar chart detailing the effect of region on promotion suggests a relatively even distribution of promotion rates across different regions. This uniformity indicates that the region may not be a decisive factor in promotional decisions within this organization. Such a pattern implies that promotions are assessed on criteria that are likely independent of regional factors, which is an important consideration for ensuring fairness in predictive modeling for promotions.

In [ ]:
# Generate crosstab
x = pd.crosstab(hr_new['region'], hr_new['is_promoted'])

plt.rcParams['figure.figsize'] = (20, 6)

# Plot
colors = plt.cm.copper(np.linspace(0, 1, 2)) 
x.div(x.sum(1).astype(float), axis = 0).plot(kind='bar', stacked=True, color=colors)

# Title and labels
plt.title('Effect of Region on Promotion', fontsize=15)
plt.xticks(rotation=45, ha='right') 
plt.xlabel('Region')
plt.ylabel('Proportion of Promotion')

plt.tight_layout()  
plt.show()

###### The area chart for the effect of a recruitment channel on promotion shows that different recruitment channels may have varying impacts on an employee’s likelihood of being promoted. It appears that employees sourced through 'referred' and 'sourcing' channels have differing promotion proportions. This could suggest that the method by which employees are recruited could correlate with their promotion prospects, potentially due to varying levels of employee fit or performance.

In [ ]:
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (15,4)
x = pd.crosstab(hr_new['recruitment_channel'], hr_new['is_promoted'])
colors = plt.cm.copper(np.linspace(0, 1, 3))
x.div(x.sum(1).astype(float), axis = 0).plot(kind = 'area', stacked = False)
plt.title('Effect of recruitment_channel on Promotion', fontsize = 15)
plt.xticks(rotation = 20)
plt.xlabel(' ')
plt.show()


#### Multivariate Analysis

###### The bar chart illustrates that average training scores—a metric of performance—differ across departments and genders, affecting promotion potential. It also shows that award recipients' training scores vary by department.

In [ ]:
# lets check the relation of Departments and Promotions when they won awards ?
plt.rcParams['figure.figsize'] = (16, 7)
sns.barplot(x='department', y='avg_training_score', data=hr_new, hue='gender')
plt.title('Chances of Promotion in each Department when they have won some Awards too')
plt.ylabel('Average Training Score', fontsize=15)
plt.xlabel('Departments', fontsize=15)
plt.show()


### 1.2 Cleanse and Transform the data

#### Handling Missing Values

###### The data inspection reveals missing values in two columns: 'education' has 2409 missing entries, and 'previous_year_rating' has 4124. Addressing these gaps is crucial for maintaining the integrity of the machine learning model.

In [ ]:
# Checking for missing values in each column
missing_values = hr_new.isnull().sum()
print(missing_values)

###### The prevalent missing 'previous_year_rating' values among employees with one year of service suggest a pattern tied to recent hiring, indicating they haven't received a rating yet. Consequently, imputing these missing ratings with a placeholder value of 0 is logical, reflecting their new status and maintaining dataset integrity for analysis.

In [ ]:
#check the rows with null values to discover if the missing values have any pattern
hr_new[hr_new['previous_year_rating'].isnull()]

In [ ]:

hypothesis = hr_new.query('length_of_service != 1')['previous_year_rating'].isna().any()

print(f' Is there any row with null values for previous_year_rating column and 1 or more for length_of_service?   {hypothesis}')

###### For 'education', imputing missing values with the mode is appropriate, reflecting the most common education level. For 'previous_year_rating', assigning a 0 to new employees with missing ratings logically differentiates them from others, avoiding confusion between no rating and poor performance. This approach upholds data consistency and readies it for precise modeling.

In [ ]:
'''
Impute missing values.
Mode values are imputed in missing education values because this is an ordinal variable.
0 will be imputed in place of the missing values in previous_year_rating.
'''
# Handling missing values in 'education'
most_frequent_education = hr_new['education'].mode()[0]  # Find the most frequent category
hr_new['education'].fillna(most_frequent_education, inplace=True)

# Handling missing values in 'previous_year_rating'
hr_new['previous_year_rating']=hr_new['previous_year_rating'].fillna(0)

###### After handling missing values, a check confirms that there are no missing values left in the dataset. This is an important milestone in data preprocessing, ensuring the dataset is now complete and ready for further analysis.

In [ ]:
# Check if there are any missing values left
missing_values_after = hr_new.isnull().sum()
print(missing_values_after)

#### Drop irrelevant features/columns

###### The 'employee_id' column has been dropped from the dataset as it does not provide any predictive value for the machine learning model. This step streamlines the dataset, ensuring that only relevant features that could influence the outcome of promotions are included for analysis.

In [ ]:
# Drop the employee_id column
hr_new.drop(['employee_id'], axis=1, inplace=True)
hr_new.head()

#### Encode Categorical Data Columns

###### Label encoding has been used to transform categorical features like 'gender', 'department', 'recruitment_channel', and 'region' into numerical values, a necessity since machine learning algorithms typically work with numerical input. This process, which assigns unique integers to each category, is essential for preparing data for modeling, especially when category order isn't crucial, ensuring the data is correctly formatted for subsequent predictive analysis.

In [ ]:
# lets use Label Encoding for Gender and Department to convert them into Numerical
from sklearn.preprocessing import LabelEncoder
le=LabelEncoder()
hr_new['gender']=le.fit_transform(hr_new['gender'])
hr_new['department']=le.fit_transform(hr_new['department'])
hr_new['recruitment_channel']=le.fit_transform(hr_new['recruitment_channel'])
hr_new['region']=le.fit_transform(hr_new['region'])

###### The 'education' variable has been encoded based on the ordinal nature of educational qualifications. This mapping reflects the assumed order of importance or level of education, assigning 'Master's & above' the highest value, followed by 'Bachelor's', and 'Below Secondary' the lowest.

In [ ]:
#education is an ordinal variable
# lets encode the education in their degree of importance
hr_new['education'] = hr_new['education'].map({"Master's & above": 2, "Bachelor's": 1, 
                                               "Below Secondary": 0}).astype(int)

###### After encoding, a check confirms there are no categorical 'object' type columns left in the dataset. The data is now fully numerical and ready for machine learning algorithms. The head() function provides a preview to ensure the encoding has been applied correctly across the dataset.

In [ ]:
# lets check whether we still have any categorical columns left after encoding
print(hr_new.select_dtypes('object').columns)
# lets check the data after encoding
hr_new.head()

In [ ]:
hr_new.describe()

#### Handling Outliers

###### Box plots for 'length_of_service' and 'avg_training_score' reveal outliers, with some employees having unusually long tenures and others displaying extreme training scores. These outliers are crucial as they might affect the model's accuracy in predicting promotions. Other numerical features, more clustered around the mean, were not subjected to similar outlier analysis.

In [ ]:
# I chose 'length_of_service' and 'avg_training_score' for outlier analysis because:
# - 'length_of_service' is a continuous variable that can theoretically have a wide range of values,
#   and outliers here may represent unusually long tenures that could skew analysis.
# - 'avg_training_score' is also a continuous variable that reflects performance scores, which
#   can vary significantly and outliers may indicate exceptionally high or low performers.
# Both features are critical to understand as they can have a significant impact on the HR analytics,
# especially when predicting employee promotion, which is the target variable in our analysis.

# Generate boxplots for 'length_of_service' with hover data
fig_length_of_service = px.box(hr_new, y='length_of_service', title='Box Plot of Length of Service', hover_data=hr_new.columns)
# Generate boxplots for 'avg_training_score' with hover data
fig_avg_training_score = px.box(hr_new, y='avg_training_score', title='Box Plot of Average Training Score', hover_data=hr_new.columns)

# Show the boxplots
fig_length_of_service.show()
fig_avg_training_score.show()

###### Outliers in the 'length_of_service' column have been removed by winsorization to mitigate the impact of extreme outliers which retains all data points thus preserving the dataset’s integrity and the validity of statistical estimates.

In [ ]:

lower_percentile = 0.05
upper_percentile = 0.95

# Perform winsorization on the 'length_of_service' column
# The limits are set as (0.05, 0.05), which means that the bottom 5% are capped at the 5th percentile and
# the top 5% are capped at the 95th percentile, thus reducing the effect of extreme outliers.
winsorized_length_of_service = winsorize(hr_new['length_of_service'], limits=(lower_percentile, 1 - upper_percentile))

# Convert the winsorized data back to a pandas series for compatibility with pandas DataFrame
hr_new['length_of_service'] = pd.Series(winsorized_length_of_service, index=hr_new.index)

# Check the new minimum and maximum values after winsorization
winsorized_min = hr_new['length_of_service'].min()
winsorized_max = hr_new['length_of_service'].max()

#### Handling Class Imbalance

###### The target variable 'is_promoted' shows a notable class imbalance, with fewer promotions than non-promotions. To ensure a balanced predictive model, techniques such as oversampling the minority class, undersampling the majority, or using methods like SMOTE are essential for effective prediction across both classes. 

In [ ]:
# Class Imbalance Check for the target variable 'is_promoted'
class_distribution = hr_new['is_promoted'].value_counts()
print("\nClass Distribution in Target Variable:\n", class_distribution)

###### The countplot shows a clear class imbalance between promoted and non-promoted employees, a disparity that could bias predictions. Addressing this with resampling techniques is crucial to avoid skewing the predictive model towards the majority class. 

In [ ]:
# Plot the class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x='is_promoted', data=hr_new)
plt.title('Class Distribution of is_promoted')
plt.show()

###### SMOTE (Synthetic Minority Over-sampling Technique) has been employed to handle the class imbalance identified in the target variable 'is_promoted'. This technique generates synthetic samples for the minority class (employees who were promoted) based on existing ones, creating a balanced dataset where the number of instances for both classes is equal. By doing so, the model is less likely to be biased towards the majority class and can improve its predictive accuracy for the minority class. The use of SMOTE is a strategic choice for datasets like this, where the class of interest (promoted employees) is underrepresented.


In [ ]:
'''
X = hr_new.drop('is_promoted', axis=1) 
y = hr_new['is_promoted'] 

# Initialize SMOTE
smote = SMOTE(random_state=42)

# Apply SMOTE
X_resampled, y_resampled = smote.fit_resample(X, y)

# Check the class distribution after applying SMOTE
print(y_resampled.value_counts())

'''

###### The countplot now displays a balanced class distribution for the target variable 'is_promoted' after applying SMOTE. This visual confirmation shows that the number of samples in both classes (promoted and not promoted) is equal, which is ideal for training a machine learning model that will not have a bias toward the majority class. The balanced dataset should help improve the reliability and validity of the predictive model’s outcomes.

In [ ]:
'''
hr_new = pd.concat([X_resampled, y_resampled], axis=1)
plt.figure(figsize=(6, 4))
sns.countplot(x='is_promoted', data=hr_new)
plt.title('Class Distribution of is_promoted')
plt.show()
'''

### 1.3 Correlation Analysis

#### Statistics Info

In [ ]:
hr_new.head()

In [ ]:
hr_new.describe()

In [ ]:
hr_new.info()

In [ ]:
# This shows the correlations between the variables
hr_new.corr()

#### Heatmap Correlation Matrix

In [ ]:
# Compute the correlation matrix
corr_matrix = hr_new.corr()

# Generate a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Matrix Heatmap')
plt.show()

###### In the correlation analysis, 'avg_training_score' and 'previous_year_rating' show positive correlations (around 0.28 and 0.24) with 'is_promoted', indicating higher performance in these areas may boost promotion chances. 'KPIs_met >80%' and 'awards_won?' also correlate positively, suggesting their influence on promotions. However, 'region', 'department', 'education', 'age', and 'length_of_service' have weaker negative correlations, implying a slight inverse effect on promotion likelihood. 'Gender' and 'no_of_trainings' display the weakest negative correlations, indicating minimal linear predictive power for promotions, possibly hinting at non-linear effects or interplays with other unmeasured factors.

In [ ]:
# Analyzing Correlation with Target
correlation_with_target = hr_new.corr()['is_promoted'].sort_values(ascending=False)
print("Correlation with 'is_promoted':")
print(correlation_with_target)

#### Feature Engineering


###### The visualization likely illustrates a positive correlation between higher ratings and promotion likelihood. This indicates that the previous year’s rating is a significant factor in promotion decisions, which aligns with intuitive expectations that better-performing employees, as judged by their ratings, would have a higher chance of being promoted.


In [ ]:
# Success rate by 'previous_year_rating'
sns.barplot(x='previous_year_rating', y='is_promoted', data=hr_new)
plt.title('Promotion Rate by Previous Year Rating')
plt.show()

###### In my analysis, I've identified key factors like previous year ratings, KPI fulfillment, and awards as crucial in predicting promotions. To consolidate these, I created the 'sum_metric', a composite score merging these elements to reflect overall performance and promotion potential more accurately.

###### This feature is based on the premise that the combined impact of these factors—previous_year_rating, KPIs_met >80%, and awards_won? — offers a stronger prediction of career progression than each component individually. 'Sum_metric' acts as a holistic performance indicator, highlighting patterns and interrelations that may be less apparent when viewed separately.

###### Also, I've developed 'training_effectiveness', the ratio of avg_training_score to no_of_trainings, to measure learning impact against training frequency. A high score with less training suggests effective learning, indicative of high performance and increased promotion likelihood.


In [ ]:
hr_new['sum_metric'] = hr_new['awards_won?']+ hr_new['previous_year_rating'] + hr_new['KPIs_met >80%']

hr_new['training_effectiveness'] = hr_new['avg_training_score'] / hr_new['no_of_trainings'].replace(0, np.nan)

###### The heatmap analysis highlights key correlations:

###### 1.	'Previous_year_rating', 'KPIs_met >80%', and 'awards_won?' positively correlate with 'is_promoted', underscoring their influence in promotion decisions.
###### 2.	A significant positive correlation between 'age' and 'length_of_service' led to the creation of 'experience_ratio', combining these factors to avoid multicollinearity and enhance prediction efficiency.
###### 3.	The engineered 'sum_metric', aggregating performance indicators, shows a strong positive correlation with 'is_promoted', proving its effectiveness as a predictive tool.
###### 4.	'Training_effectiveness', derived from 'avg_training_score' and 'no_of_trainings', also correlates positively with promotions, indicating the value of efficient learning in career advancement.


In [ ]:
# Compute the correlation matrix
corr_matrix = hr_new.corr()

# Generate a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Matrix Heatmap')
plt.show()

###### The decision to remove 'age', 'length_of_service', 'no_of_trainings', 'education', and 'previous_year_rating' can be succinctly justified:

###### •	Age ('age') and Gender (‘gender’): The correlation with 'is_promoted' is weak, suggesting age is not a strong predictor of promotions.
###### •	Length of Service ('length_of_service'): The moderate correlation does not sufficiently justify its predictive relevance for promotions.
###### •	Number of Trainings ('no_of_trainings'): The negative correlation indicates that more trainings do not necessarily lead to promotions, possibly reflecting remedial training rather than advancement.
###### •	Education ('education'): Beyond meeting the role's minimum requirements, additional educational qualifications do not significantly enhance the likelihood of promotion.
###### •	Previous Year Rating ('previous_year_rating'): A high correlation with 'sum_metric' suggests redundancy, and its removal mitigates multicollinearity, enhancing model simplicity and focus on composite performance metrics.


In [ ]:
hr_new.drop(['age', 'length_of_service', 'education', 
                     'gender', 'no_of_trainings', 'previous_year_rating'], axis=1, inplace=True)

###### In this visualization, I noticed varying spreads across features. For instance, 'avg_training_score' shows a broad distribution, suggesting diversity in employee performance scores.

In [ ]:
# List of all numerical features after encoding and scaling
numerical_features = [col for col in hr_new.columns]

# Plotting boxplots for each numerical feature
plt.figure(figsize=(15, 10))
sns.boxplot(data=hr_new[numerical_features])
plt.xticks(rotation=90)  # Rotate the feature names for better readability
plt.title('Boxplots of all numerical features')
plt.show()

###### In this step, 'sum_metric', 'avg_training_score', and 'training_effectiveness' have been scaled using StandardScaler, which normalizes the distribution of these features with a mean of zero and standard deviation of one.


In [ ]:
numerical_features_to_scale = ['sum_metric', 'avg_training_score', 'training_effectiveness']

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler to the numerical features and transform them
hr_new[numerical_features_to_scale] = scaler.fit_transform(hr_new[numerical_features_to_scale])

sns.boxplot(data=hr_new)
plt.title('Boxplots of Scaled Numerical Features')
plt.show()


### 1.4 Export the data

In [ ]:
hr_new.to_csv('hr_data_new.csv', index=False)

## 2. Airbnb

### 2.1 Load and Explore the data

#### Statistical Analysis

In [ ]:
# Load the dataset
listing_data = pd.read_csv('listings.csv')

# Create a copy of the DataFrame for further analysis
listing_new = listing_data.copy()

###### The dataset was observed to have 7,987 entries each with 16 attributes. This can help me understand the scale and scope of the data

In [ ]:
listing_new.shape

###### The dataset offers a window into the varied Airbnb market, with listings spanning multiple neighborhoods and prices from $69 to $550, encompassing private rooms to entire homes.

In [ ]:
listing_new.head()

In [ ]:
listing_new.tail()

###### The dataframe comprises 7,907 entries, each characterized by 16 variables. All columns are almost complete with non-null entries except for last_review and reviews_per_month, indicating a need for further scrutiny regarding missing values.

In [ ]:
listing_new.info()

###### The statistical summary of the dataset highlights the diversity of Airbnb offerings, with an average price of around $169.34, hinting at the influence of high-end listings.

In [ ]:
listing_new.describe()

#### Numerical Data Analysis

###### A dataframe was created to focus on numerical features essential for the regression problem, enabling targeted analysis of key variables such as price and minimum nights.

In [ ]:
listing_num = listing_new.select_dtypes(['int64', 'float64']).copy()
listing_num.head()

In [ ]:
listing_num.describe()

###### Histograms of listing_num highlight skewness and distribution patterns, with price and minimum nights being right skewed, indicating a need for normalization. The availability throughout the year and bimodal longitude suggests varied listing availability and location clusters, informing potential data transformations for optimal model performance.

In [ ]:
# Assuming listing_num is your DataFrame with numerical columns
numeric_cols = listing_num.columns
num_rows = (len(numeric_cols) + 1) // 2  # Simplified calculation for rows

fig, axes = plt.subplots(num_rows, 2, figsize=(16, 5 * num_rows))
axes = axes.flatten()  # Flatten the axes array

# Plot histograms
for i, col in enumerate(numeric_cols):
    sns.histplot(listing_num[col], ax=axes[i], kde=True)
    axes[i].set_title(f'Histogram of {col}')

# Hide unused subplots if the number of numeric columns is odd
if len(numeric_cols) % 2:
    axes[-1].axis('off')

plt.tight_layout()

###### Boxplots for numerical features reveal outliers and data spread, crucial for identifying extremes in price and minimum nights that might impact model accuracy. Variations in reviews and availability suggest differences in listing popularity and seasonality, important for data preprocessing.

In [ ]:
# Calculate the number of rows needed for 2 columns
num_rows = (len(numeric_cols) + 1) // 2

# Create a subplot grid with 2 columns
fig, axes = plt.subplots(num_rows, 2, figsize=(20, 6 * num_rows))
axes = axes.flatten()  # Flatten the axes array

# Plot box plots
for i, col in enumerate(numeric_cols):
    sns.boxplot(x=listing_new[col], ax=axes[i])
    axes[i].set_title(f'Box Plot of {col}')

# Hide unused subplot if the number of numeric columns is odd
if len(numeric_cols) % 2:
    axes[-1].axis('off')

plt.tight_layout()

#### Categorical Data

###### A new dataframe listing_cat was derived by selecting features of type 'object' from listing_new, which represent the categorical data in the dataset. Understanding the nature of these non-numerical entries is essential for feature engineering and preparing the data for algorithms that can only interpret numeric inputs.

In [ ]:
listing_cat = listing_new.select_dtypes(['object']).copy()
listing_cat.head()

In [ ]:
listing_cat.describe()

###### The nunique() function highlighted the diversity in the listing_cat dataset, with a vast array of unique listing names and hosts. In contrast, 'neighbourhood_group' and 'room_type' categories have fewer unique values, suggesting a concentrated distribution that could significantly influence the model. 'last_review' dates also show variety. This cardinality check is crucial for effective categorical feature encoding and model refinement.

In [ ]:
listing_cat.nunique()

###### Count plots for listing_cat reveal a dominance of private rooms in the dataset, with entire homes and few shared rooms. Neighbourhood distribution suggests certain areas have higher Airbnb activity, and the Central Region emerges as a hotspot in neighbourhood_group analysis, informing pricing and marketing strategies.

In [ ]:
# Frequency of different categories in 'room_type', 'neighbourhood', 'neighbourhood_group'
fig, axes = plt.subplots(3, 1, figsize=(14, 18))

# Plot 'room_type' frequency
sns.countplot(y=listing_new['room_type'], ax=axes[0], color='skyblue')
axes[0].set_title('Frequency of Room Types')
axes[0].xaxis.set_major_locator(ticker.MultipleLocator(500))
axes[0].xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Plot 'neighbourhood' frequency
neighbourhood_order = listing_new['neighbourhood'].value_counts().index
sns.countplot(y=listing_new['neighbourhood'], ax=axes[1], order=neighbourhood_order, color='skyblue')
axes[1].set_title('Frequency of Neighbourhoods')
axes[1].xaxis.set_major_locator(ticker.MultipleLocator(100))
axes[1].xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

# Plot 'neighbourhood_group' frequency
sns.countplot(y=listing_new['neighbourhood_group'], ax=axes[2], color='skyblue')
axes[2].set_title('Frequency of Neighbourhood Groups')
axes[2].xaxis.set_major_locator(ticker.MultipleLocator(500))
axes[2].xaxis.set_major_formatter(ticker.StrMethodFormatter('{x:,.0f}'))

for ax in axes:
    ax.xaxis.grid(True)
    ax.set_ylabel('') 

plt.tight_layout()
plt.show()

###### The pie chart illustrates that private rooms are the most common listing type in the dataset, followed by entire homes, with shared rooms making up a smaller fraction, providing a clear visual of market preferences in the Airbnb space.

#### Univariate Analysis

In [ ]:
roomdf = listing_new.groupby('room_type').size()/listing_new['room_type'].count()*100
labels = roomdf.index
values = roomdf.values

# Use `hole` to create a donut-like pie chart
fig = go.Figure(data=[go.Pie(labels=labels, values=values, hole=.6)])
fig.show()

###### A bivariate analysis was conducted focusing on the relationship between price and neighbourhood groups for listings under $250. The boxplot illustrates the variation in median prices and the spread within different neighbourhood groups. It is noticeable that the Central Region has a higher median price, along with a greater range of prices and outliers, which suggests it may be a more premium or varied market compared to others. The East and North-East Regions show more compact distributions, indicating less variability in pricing. 

#### Bivariate Analysis

In [ ]:
ng = listing_new[listing_new.price <250]
plt.figure(figsize=(10,6))
sns.boxplot(y="price",x ='neighbourhood_group' ,data = ng)
plt.title("neighbourhood_group price distribution < 250")
plt.show()

###### The countplot shows private rooms as the most common accommodation, especially in the Central Region, with entire homes next in prevalence and shared rooms least common, indicating regional demand and listing availability trends, which could be indicative of lower demand or fewer listings in this category.

In [ ]:
plt.figure(figsize=(10,6))
sns.countplot(x = 'room_type',hue = "neighbourhood_group",data = listing_new)
plt.title("Room types occupied by the neighbourhood_group")
plt.show()

###### The catplot reveals that entire homes/apartments generally have higher prices than private and shared rooms, with notable price variability and premium offerings in the former. Shared rooms display more uniform pricing, indicating a consistent strategy for this accommodation type

In [ ]:
#catplot room type and price
plt.figure(figsize=(10,6))
sns.catplot(x="room_type", y="price", data=listing_new);
plt.ioff()

#### Multivariate Analysis

###### The implementation of a Folium map with a MarkerCluster is a sophisticated multivariate analysis technique, allowing for the exploration of geographical data alongside other variables. In the Airbnb context, this map visually clusters listings and reveals patterns of distribution across different neighborhoods, providing an interactive way to understand how the location correlates with other factors like pricing or room type.

In [ ]:
Long=103.91492
Lat=1.32122
mapdf1=folium.Map([Lat,Long],zoom_start=10,)

mapdf1_rooms_map=plugins.MarkerCluster().add_to(mapdf1)

for lat,lon,label in zip(listing_new.latitude,listing_new.longitude,listing_new.name):
    folium.Marker(location=[lat,lon],icon=folium.Icon(icon='home'),popup=label).add_to(mapdf1_rooms_map)
mapdf1.add_child(mapdf1_rooms_map)

mapdf1

###### A scatter plot was created to visualize the availability of Airbnb listings across geographical locations, with the color intensity representing the number of days available for booking in a year. This visual tool is excellent for identifying patterns of availability that may correspond to different areas. For example, areas with higher availability might suggest less demand or a higher turnover of properties, whereas those with lower availability could indicate higher demand or listings that are more frequently occupied.

In [ ]:
plt.figure(figsize=(10,6))
plt.scatter(listing_new.longitude, listing_new.latitude, c=listing_new.availability_365, 
            cmap='spring', edgecolor='black', linewidth=1\
            , alpha=1)

cbar = plt.colorbar()
cbar.set_label('availability_365')

plt.show()

###### The scatter plot shows the relationship between the geographic location of Airbnb listings under $500, their price, and yearly availability. Price is indicated by color intensity, highlighting areas with either affordable or premium listings. It reveals that central areas have more expensive listings, likely due to proximity to attractions or business districts, while affordable options are spread out, suggesting residential or less tourist-focused areas. This analysis aids in identifying regions with specific pricing trends.

In [ ]:
plt.figure(figsize=(10,6))
sub_6=listing_new[listing_new.price<500]
viz_4=sub_6.plot(kind='scatter', x='longitude',y='latitude',label='availability_365',c='price',cmap=plt.get_cmap('jet'),
                 colorbar=True,alpha=0.4,figsize=(10,10))
viz_4.legend()
plt.ioff()

plt.show()

###### The interactive scatter mapbox plot, designed using Plotly, is an advanced analytical tool that maps the pricing data of Airbnb listings onto their geographical coordinates. The color-coded price points provide a gradient visualization, where the color intensity correlates with the listing price, thus offering a heatmap perspective of the pricing landscape.

In [ ]:
fig = px.scatter_mapbox(
    listing_new,
    lat="latitude",
    lon="longitude",
    color="price",
    color_continuous_scale=px.colors.sequential.Viridis,
    range_color=[65, 300], # (25th percentile) and (a bit higher 75th percentile)
    size_max=15,
    zoom=10
)

fig.update_layout(mapbox_style="open-street-map")
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.show()


### 2.2 Cleanse and Transform the data

#### Dropping Irrelevant Features/Columns

###### The calculated correlation of approximately 0.627 between 'reviews_per_month' and 'number_of_reviews' indicates a moderate to strong positive relationship.

In [ ]:
# Calculate the correlation between the two columns
correlation = listing_new['reviews_per_month'].corr(listing_new['number_of_reviews'])

print("Correlation:", correlation)

###### The features 'name', 'id', and 'host_name' were removed for being unique identifiers without predictive value. 'last_review' requires further processing to be useful, while 'host_id' is also non-predictive. 'reviews_per_month' was dropped to avoid multicollinearity with 'number_of_reviews', retaining a more static measure of listing popularity.

In [ ]:
# Columns to be dropped
listing_new.drop(['name','id','host_name','last_review', 'host_id', 'reviews_per_month'], axis=1, inplace=True)

# Verify the changes
listing_new.head()

#### Handling Missing Values

###### The output confirms that the dataset is free of missing values across all examined features. 

In [ ]:
# Checking for missing values in each column
missing_values = listing_new.isnull().sum()
print(missing_values)

#### Narrowing Down the Dataset

###### I have identified the Central Region as having the median price closest to the overall median for all listings. This strategic choice is predicated on the Central Region's price point. A median value is a robust indicator of a typical listing's price, less sensitive to extreme values than an average. Hence, the Central Region, aligning closely with this central tendency, provides a quintessential segment for studying market-wide pricing behaviors. Analyzing this area can yield insights that are likely to be generalizable across the entire dataset, making it an ideal candidate for in-depth examination and model training.

In [ ]:
# Calculate the overall median price
overall_median_price = listing_new['price'].median()

# Calculate the median price for each neighbourhood group
median_price_by_region = listing_new.groupby('neighbourhood_group')['price'].median()

# Find the neighbourhood group with the median price closest to the overall median
closest_to_overall_median = median_price_by_region.sub(overall_median_price).abs().idxmin()

print(f"The neighbourhood group with the median price closest to the overall median ({overall_median_price}) is: {closest_to_overall_median}")


###### The dataset has been refined to exclusively encompass listings from the Central Region, following a data-driven selection criterion. This region not only mirrors the median pricing structure of the entire market but also boasts a substantial volume of data, which is critical for ensuring the statistical significance and reliability of any subsequent analysis.

In [ ]:
# Filter the dataset to include only listings from the Central Region
central_region_listings = listing_new[listing_new['neighbourhood_group'] == 'Central Region']

listing_new = central_region_listings.copy()

listing_new.head()

###### I visualized the Central Region's price distribution using a histogram, revealing most listings concentrated in the lower price range and a long tail towards higher prices. This helps identify a central pricing cluster, avoiding skew from high-value outliers.

In [ ]:
# Price distribution in the Central Region
plt.figure(figsize=(12, 6))
sns.histplot(listing_new['price'], kde=True)
plt.title('Price Distribution in Central Region')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()


###### I applied an interquartile range (IQR) filter to the Central Region's pricing data, setting bounds to the 5th and 95th percentiles. This approach removed outliers and anomalies, focusing the dataset on the central pricing distribution, which more accurately represents typical market conditions and average consumer experiences. The resulting smoother histogram is advantageous for developing a predictive model without skew from extreme values.

In [ ]:
# Calculate the first and third quartile
Q1 = listing_new['price'].quantile(0.25)
Q3 = listing_new['price'].quantile(0.75)
IQR = Q3 - Q1

# Set the upper bound to the 95th percentile to retain more data
upper_bound = listing_new['price'].quantile(0.95)

# Set the lower bound to the 5th percentile to remove extremely low prices that could be anomalies
lower_bound = listing_new['price'].quantile(0.05)

# Apply the filter
filtered_listings = listing_new[(listing_new['price'] >= lower_bound) & (listing_new['price'] <= upper_bound)]

# Check the new price distribution
plt.figure(figsize=(12, 6))
sns.histplot(filtered_listings['price'], kde=True)
plt.title('Filtered Price Distribution in Central Region')
plt.xlabel('Price')
plt.ylabel('Frequency')
plt.show()

#### Encoding Categorical Features

###### I have utilized Label Encoding to transform categorical features into a numerical format. Label Encoding assigns a unique integer to each category. For instance, different neighbourhood groups and specific neighbourhoods are encoded to facilitate their inclusion in the predictive model, where they can be treated as quantitative variables. I have also applied a manual mapping for 'room_type' to ensure a consistent and understandable transformation.

In [ ]:
# lets use Label Encoding for Gender and Department to convert them into Numerical
le=LabelEncoder()
listing_new['neighbourhood_group']=le.fit_transform(listing_new['neighbourhood_group'])
listing_new['neighbourhood']=le.fit_transform(listing_new['neighbourhood'])
listing_new['room_type']=listing_new['room_type'].map({'Entire home/apt':2,'Private room':1,'Shared room':0})

In [ ]:
# lets check whether we still have any categorical columns left after encoding
print(listing_new.select_dtypes('object').columns)

#### Handling Outliers

###### I have plotted box plots for each numerical feature to visually inspect for outliers. Observing these plots allows me to identify which features contain outliers that may need to be addressed.

In [ ]:
numerical_features=['latitude','longitude','price','minimum_nights','number_of_reviews',
                    'calculated_host_listings_count','availability_365']
plt.figure(figsize=(12, 10))

# Loop to create a subplot for each column
for j in range(1, 8):
    plt.subplot(2, 4, j)
    sns.boxplot(y=listing_new[numerical_features[j-1]])

plt.tight_layout()
plt.show()

###### After visual inspection using boxplots, I have identified outliers using the Interquartile Range (IQR) method for each numerical feature. The IQR is the range between the first quartile (25th percentile) and the third quartile (75th percentile). By calculating 1.5 times the IQR above and below this range, I have established thresholds for detecting outliers.

In [ ]:
# Calculate IQR for each numeric column
for col in numerical_features:
    Q1 = listing_new[col].quantile(0.25)
    Q3 = listing_new[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR

    # Identify outliers
    outliers = listing_new[(listing_new[col] < lower_bound) | (listing_new[col] > upper_bound)]
    print(f"Outliers detected in '{col}': {len(outliers)}")

###### In this scatter plot, I've depicted the geographical distribution of listings, with a focus on identifying outliers based on latitude and longitude coordinates. The condition I've set identifies listings outside typical geographical bounds as outliers, represented here in red. This visualization is here does not show any outliers for latitude and longitude.

In [ ]:
outlier_condition = (listing_new['longitude'] > 103.9) & (listing_new['latitude'] > 1.34)

# Plotting the coordinates
plt.figure(figsize=(10, 6))
plt.scatter(listing_new['longitude'], listing_new['latitude'], alpha=0.4, label='Normal')
plt.scatter(listing_new.loc[outlier_condition, 'longitude'], listing_new.loc[outlier_condition, 'latitude'], color='red', 
            label='Outliers') 
plt.title('Scatter plot of listings')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.legend()
plt.show()

###### In this analysis, I used log transformation for 'price' and 'minimum_nights' to address their right-skewed distributions. For 'minimum_nights', values were capped at the 95th percentile instead of removal, balancing outlier mitigation and data retention. No adjustments were made to 'number_of_reviews', as its right-skewed nature is typical for count data and does not necessarily indicate quality issues. 'Calculated_host_listings_count' underwent winsorization at the 95th percentile to limit outlier effects, offering a middle ground between removing and leaving outliers untreated, thus preserving data integrity while reducing skewness impact.

In [ ]:
# Log Transformation for 'price' to reduce skewness and normalize distribution
# Reasoning: Log transformation is useful for right-skewed distributions to reduce the impact of extreme values
# listing_new['price'] = np.log1p(listing_new['price'])

# Cap 'minimum_nights' at the 95th percentile to reduce the impact of extreme outliers
# Reasoning: Capping reduces the influence of extreme high values while retaining all data points
min_nights_95th_percentile = listing_new['minimum_nights'].quantile(0.95)
listing_new['minimum_nights'] = listing_new['minimum_nights'].apply(lambda x: min(x, min_nights_95th_percentile))

# Log Transformation for 'minimum_nights' as an alternative to capping
# Reasoning: Log transformation can also handle right-skewed count data effectively
listing_new['minimum_nights'] = np.log1p(listing_new['minimum_nights'])

# No action taken for 'number_of_reviews'
# Reasoning: Right-skewed count data is common and does not necessarily indicate outliers that need to be handled

## Applying Winsorizing to the 'calculated_host_listings_count' column
# Reasoning:
# - To limit the influence of extreme outliers which might be due to data entry errors or unusual circumstances
# - To retain as much data as possible instead of discarding outliers

# Winsorize the data by setting values above the 95th percentile to the value at the 95th percentile
listing_new['calculated_host_listings_count'] = mstats.winsorize(listing_new['calculated_host_listings_count'], limits=[0, 0.05])

### 2.3 Correlation Analysis

#### Statistical Info

In [ ]:
listing_new.head()

In [ ]:
listing_new.describe()

In [ ]:
listing_new.corr()

#### Heatmap Correlation Matix

In [ ]:
# Compute the correlation matrix
corr_matrix = listing_new.corr()

# Generate a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Matrix Heatmap')
plt.show()

###### In the Airbnb dataset analysis, room type shows a strong positive correlation (0.652920) with rental price, indicating a significant impact on pricing. Hosts with more listings have moderately higher prices (correlation: 0.149332). Property availability throughout the year has a slight positive correlation (0.057073) with price, suggesting that more available properties might be priced higher. In contrast, geographic coordinates (longitude and latitude) display a weak negative correlation, implying a lesser impact on pricing. Properties with longer minimum stays and more reviews are associated with slightly lower prices, both showing minor negative correlations (-0.056314 and -0.057148, respectively). 'Neighbourhood' shows a negligible correlation, and 'neighbourhood_group' has a non-applicable correlation, indicating these factors may not significantly influence price independently.

In [ ]:
# Analyzing Correlation with Target
correlation_with_target = listing_new.corr()['price'].sort_values(ascending=False)
print("Correlation with 'price':")
print(correlation_with_target)

###### 1.	The 'Price Per Minimum Stay' metric quantifies the total cost for the shortest possible stay, aiding guests in evaluating the value of a listing. This is especially useful for guests interested in shorter stays, as it highlights the cost-duration trade-off.

###### 2.	The 'Competitive Pricing Index' compares a host's pricing to the neighborhood average, reflecting market saturation and competitive stance. It's important for modeling a host's pricing strategy and market impact, aiding in setting optimal prices for high occupancy and revenue.

###### 3.	'Location Cluster' groups listings based on geographic coordinates, highlighting areas with similar features that affect desirability and price. This feature is essential for models to analyze location-based pricing trends and identify in-demand areas.


#### Feature Engineering

In [ ]:
listing_new['price_per_minimum_stay'] = listing_new['minimum_nights'] * listing_new['price']

# Calculate the average price for each neighborhood
average_price_per_neighbourhood = listing_new.groupby('neighbourhood')['price'].mean().to_dict()

# Calculate the competitive pricing index for each listing
listing_new['competitive_pricing_index'] = listing_new.apply(
    lambda row: row['calculated_host_listings_count'] / average_price_per_neighbourhood[row['neighbourhood']], axis=1)

from sklearn.cluster import KMeans

# Assuming we want 10 clusters
kmeans = KMeans(n_clusters=10, random_state=42)

# Fit the model and predict the cluster for each listing
listing_new['location_cluster'] = kmeans.fit_predict(listing_new[['latitude', 'longitude']])

In [ ]:
from sklearn.cluster import KMeans

# Assuming we want 10 clusters
kmeans = KMeans(n_clusters=10, random_state=42)

# Fit the model and predict the cluster for each listing
listing_new['location_cluster'] = kmeans.fit_predict(listing_new[['latitude', 'longitude']])

In [ ]:
# Neighbourhood popularity (count of listings in each neighbourhood)

neighbourhood_popularity = listing_new.groupby('neighbourhood')['neighbourhood'].transform('count')
listing_new['neighbourhood_popularity'] = neighbourhood_popularity

In [ ]:
# High availability flag (1 if availability_365 is above the median, else 0)
median_availability = listing_new['availability_365'].median()
listing_new['high_availability'] = (listing_new['availability_365'] > median_availability).astype(int)

In [ ]:
room_type_dummies = pd.get_dummies(listing_new['room_type'], prefix='room_type', drop_first=True)

# Concatenate the one-hot encoded columns back to the original DataFrame
listings_df_encoded = pd.concat([listing_new, room_type_dummies], axis=1)

# Optionally, drop the original 'room_type' column if you no longer need it
listings_df_encoded = listings_df_encoded.drop('room_type', axis=1)

In [ ]:
from geopy.distance import geodesic

# Load the MRT stations data
mrt_stations_df = pd.read_csv('Cleaned_MRT_Stations.csv')

# Convert the MRT stations DataFrame into a dictionary
mrt_stations = {row['Station Name']: (row['Latitude'], row['Longitude']) for index, row in mrt_stations_df.iterrows()}

# Function to calculate the distance to the nearest MRT station
def distance_to_nearest_mrt(latitude, longitude, mrt_stations):
    min_distance = float('inf')

    for _, coords in mrt_stations.items():
        distance = geodesic((latitude, longitude), coords).kilometers
        if distance < min_distance:
            min_distance = distance

    return min_distance

# Assume your Airbnb data is in a DataFrame named listings_df_encoded
# Apply the function to calculate only the distance to the nearest MRT
listings_df_encoded['distance_from_mrt'] = listings_df_encoded.apply(lambda row: distance_to_nearest_mrt(row['latitude'], row['longitude'], mrt_stations), axis=1)

# Displaying the first few rows to confirm the new column is added correctly
print(listings_df_encoded[['latitude', 'longitude', 'distance_from_mrt']].head())


In [ ]:
# Distance from MRT to categories
listings_df_encoded['distance_category'] = pd.cut(listings_df_encoded['distance_from_mrt'], bins=[0, 0.5, 1, 1.5, np.inf], labels=['Very Close', 'Close', 'Moderate', 'Far'])

In [ ]:
distance_category_dummies = pd.get_dummies(listings_df_encoded['distance_category'], prefix='distance_category', drop_first=True)

# Concatenate the one-hot encoded columns back to the original DataFrame
final_df_encoded = pd.concat([listings_df_encoded, distance_category_dummies], axis=1)

# Optionally, drop the original 'distance_category' column if you no longer need it
final_df_encoded = final_df_encoded.drop('distance_category', axis=1)

###### Key takeaways from the heatmap include:

###### 1.	Room Type and Price: There's a highly positive correlation, indicating room type's significant impact on pricing.

###### 2.	Price Per Minimum Stay: Demonstrates a strong positive correlation with price, highlighting its relevance as a financial metric for guests.

###### 3.	Competitive Pricing Index: Exhibits moderate correlation with various features, suggesting it's influenced by multiple listing aspects.

###### 4.	Location Cluster: Although not strongly correlated with price, it shows a better correlation than latitude and longitude alone.


In [ ]:
# Compute the correlation matrix
corr_matrix = final_df_encoded.corr()

# Generate a heatmap
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap='coolwarm')
plt.title('Correlation Matrix Heatmap')
plt.show()

###### 1.	Neighbourhood Group: Removed due to its lack of numerical correlation, suggesting little variability or influence on price, making it irrelevant for the predictive model.

###### 2.	Minimum Nights: Dropped in favor of 'price_per_minimum_stay' due to high collinearity and its more direct economic relevance for guests, avoiding multicollinearity in the model.

###### 3.	Number of Reviews: Removed because of its slight negative correlation with rental price and its focus on popularity rather than quality or desirability affecting price.

###### 4.	Latitude and Longitude: Replaced by the 'location_cluster' feature, which condenses locational attributes into a more model-friendly format, avoiding the complexity of raw geographical data.


In [ ]:
final_df_encoded.drop(['latitude', 'longitude'], axis=1, inplace=True)

###### The boxplot visualization reveals significant variance and outliers in certain features, impacting the performance of algorithms sensitive to data value ranges. Features like 'calculated_host_listings_count', 'price_per_minimum_stay', 'availability_365', and 'competitive_pricing_index' show a broad range of values and outliers, indicating a need for scaling to equalize their impact in the predictive model. Conversely, 'room_type' and 'location_cluster' display less variation and fewer outliers, suggesting less need for scaling.

In [ ]:
# List of all numerical features after encoding and scaling
numerical_features = [col for col in listing_new.columns]

# Plotting boxplots for each numerical feature
plt.figure(figsize=(15, 10))
sns.boxplot(data=listing_new[numerical_features])
plt.xticks(rotation=90)  # Rotate the feature names for better readability
plt.title('Boxplots of all numerical features')
plt.show()

###### Selected numerical features in the Airbnb dataset—'availability_365', 'price_per_minimum_stay', 'competitive_pricing_index', and 'calculated_host_listings_count'—were chosen for scaling using StandardScaler due to varying scales and outliers:
###### •	'availability_365': Scaled to normalize its full-year range, preventing disproportionate influence compared to features with smaller numerical ranges.
###### •	'price_per_minimum_stay': Important to scale as it merges price and minimum nights into a wide value range, which could otherwise dominate the learning process.
###### •	'competitive_pricing_index': Scaling this derived metric ensures normalized composite values for equal weighting in predictive analysis.
###### •	'calculated_host_listings_count': Scaled to normalize data for hosts with many listings, avoiding bias towards those with extensive portfolios.


In [ ]:
numerical_features_to_scale = ['availability_365', 
                               'calculated_host_listings_count']

# Initialize the scaler
scaler = StandardScaler()

# Fit the scaler to the numerical features and transform them
listing_new[numerical_features_to_scale] = scaler.fit_transform(listing_new[numerical_features_to_scale])

#increase size
plt.figure(figsize=(15, 10))
sns.boxplot(data=listing_new)
plt.xticks(rotation=90)  # Rotate the feature names for better readability
plt.title('Boxplots of Scaled Numerical Features')
plt.show()

### 2.4 Export the data

In [ ]:
final_df_encoded.to_csv('listings_new.csv', index=False)